In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ขั้นตอนที่ 1: กำหนดโมเดล Pydantic สำหรับผลลัพธ์ที่มีโครงสร้าง

โมเดลเหล่านี้กำหนด **สคีมา** ที่ตัวแทนจะส่งกลับ การใช้ `response_format` กับ Pydantic จะช่วยให้:
- ✅ การดึงข้อมูลแบบปลอดภัยด้วยชนิดข้อมูล
- ✅ การตรวจสอบอัตโนมัติ
- ✅ ไม่มีข้อผิดพลาดในการแยกวิเคราะห์จากการตอบกลับข้อความอิสระ
- ✅ การกำหนดเส้นทางตามเงื่อนไขที่ง่ายขึ้นโดยอิงจากฟิลด์


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ขั้นตอนที่ 2: สร้างเครื่องมือจองโรงแรม

เครื่องมือนี้คือสิ่งที่ **availability_agent** จะเรียกใช้เพื่อตรวจสอบว่ามีห้องว่างหรือไม่ เราใช้ตัวตกแต่ง `@ai_function` เพื่อ:
- แปลงฟังก์ชัน Python ให้เป็นเครื่องมือที่สามารถเรียกใช้ผ่าน AI ได้
- สร้างสคีมา JSON อัตโนมัติสำหรับ LLM
- จัดการการตรวจสอบความถูกต้องของพารามิเตอร์
- เปิดใช้งานการเรียกอัตโนมัติโดยเอเย่นต์

สำหรับเดโม่นี้:
- **สตอกโฮล์ม, ซีแอตเทิล, โตเกียว, ลอนดอน, อัมสเตอร์ดัม** → มีห้องว่าง ✅
- **เมืองอื่น ๆ ทั้งหมด** → ไม่มีห้องว่าง ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ขั้นตอนที่ 3: กำหนดฟังก์ชันเงื่อนไขสำหรับการกำหนดเส้นทาง

ฟังก์ชันเหล่านี้จะตรวจสอบการตอบกลับของตัวแทนและกำหนดเส้นทางที่ต้องเลือกในเวิร์กโฟลว์

**รูปแบบสำคัญ:**
1. ตรวจสอบว่าข้อความเป็น `AgentExecutorResponse` หรือไม่
2. วิเคราะห์ผลลัพธ์ที่มีโครงสร้าง (โมเดล Pydantic)
3. ส่งกลับ `True` หรือ `False` เพื่อควบคุมการกำหนดเส้นทาง

เวิร์กโฟลว์จะประเมินเงื่อนไขเหล่านี้บน **edges** เพื่อเลือก executor ถัดไปที่ต้องเรียกใช้


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## ขั้นตอนที่ 4: สร้าง Custom Display Executor

Executors คือส่วนประกอบของ workflow ที่ทำการแปลงข้อมูลหรือผลข้างเคียง เราใช้ตัวตกแต่ง `@executor` เพื่อสร้าง executor แบบกำหนดเองที่แสดงผลลัพธ์สุดท้าย

**แนวคิดหลัก:**
- `@executor(id="...")` - ลงทะเบียนฟังก์ชันเป็น workflow executor
- `WorkflowContext[Never, str]` - ทิปประเภทสำหรับอินพุต/เอาท์พุต
- `ctx.yield_output(...)` - ส่งผลลัพธ์สุดท้ายของ workflow


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## ขั้นตอนที่ 5: โหลดตัวแปรสภาพแวดล้อม

กำหนดค่าลูกค้า LLM ตัวอย่างนี้ใช้กับ:
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — แนะนำ)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ขั้นตอนที่ 6: สร้างเอเจนต์ AI พร้อมผลลัพธ์ที่มีโครงสร้าง

เราสร้าง **เอเจนต์เฉพาะทางสามตัว** แต่ละตัวถูกห่อหุ้มด้วย `AgentExecutor`:

1. **availability_agent** - ตรวจสอบความพร้อมของโรงแรมโดยใช้เครื่องมือ
2. **alternative_agent** - แนะนำเมืองทางเลือก (เมื่อไม่มีห้องพัก)
3. **booking_agent** - กระตุ้นให้จอง (เมื่อมีห้องว่าง)

**คุณสมบัติสำคัญ:**
- `tools=[hotel_booking]` - ให้เครื่องมือกับเอเจนต์
- `response_format=PydanticModel` - บังคับให้แสดงผล JSON ที่เป็นโครงสร้าง
- `AgentExecutor(..., id="...")` - ห่อหุ้มเอเจนต์เพื่อใช้ในเวิร์กโฟลว์


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ขั้นตอนที่ 7: สร้าง Workflow ด้วย Conditional Edges

ตอนนี้เราจะใช้ `WorkflowBuilder` เพื่อสร้างกราฟพร้อมการนำทางตามเงื่อนไข:

**โครงสร้าง Workflow:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**เมธอดสำคัญ:**
- `.set_start_executor(...)` - กำหนดจุดเริ่มต้น
- `.add_edge(from, to, condition=...)` - เพิ่ม conditional edge
- `.build()` - สรุป workflow


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ขั้นตอนที่ 8: รันกรณีทดสอบ 1 - เมืองที่ไม่มีห้องว่าง (ปารีส)

ให้เราทดสอบเส้นทาง **ไม่มีห้องว่าง** โดยการขอข้อมูลโรงแรมในปารีส (ซึ่งไม่มีห้องว่างในจำลองของเรา)


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ขั้นตอนที่ 9: รันกรณีทดสอบ 2 - เมืองที่มีการว่าง (Stockholm)

ตอนนี้เรามาทดสอบเส้นทาง **availability** โดยการขอโรงแรมใน Stockholm (ซึ่งมีห้องว่างในแบบจำลองของเรา)


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ข้อสรุปสำคัญและขั้นตอนถัดไป

### ✅ สิ่งที่คุณได้เรียนรู้:

1. **รูปแบบ WorkflowBuilder**
   - ใช้ `.set_start_executor()` เพื่อกำหนดจุดเริ่มต้น
   - ใช้ `.add_edge(from, to, condition=...)` สำหรับการกำหนดเส้นทางตามเงื่อนไข
   - เรียก `.build()` เพื่อสรุปเวิร์กโฟลว์

2. **การกำหนดเส้นทางตามเงื่อนไข**
   - ฟังก์ชันเงื่อนไขตรวจสอบ `AgentExecutorResponse`
   - วิเคราะห์ผลลัพธ์ที่มีโครงสร้างเพื่อการตัดสินใจเส้นทาง
   - คืนค่า `True` เพื่อเปิดใช้งานเส้นทาง, `False` เพิ่อข้ามเส้นทางนั้น

3. **การผสานเครื่องมือ**
   - ใช้ `@ai_function` เพื่อแปลงฟังก์ชัน Python เป็นเครื่องมือ AI
   - Agents เรียกใช้เครื่องมือโดยอัตโนมัติเมื่อจำเป็น
   - เครื่องมือคืนค่า JSON ที่ agents สามารถวิเคราะห์ได้

4. **ผลลัพธ์ที่มีโครงสร้าง**
   - ใช้โมเดล Pydantic สำหรับการดึงข้อมูลอย่างปลอดภัยตามชนิด
   - ตั้ง `response_format=MyModel` เมื่อสร้าง agents
   - วิเคราะห์การตอบกลับด้วย `Model.model_validate_json()`

5. **Executor แบบกำหนดเอง**
   - ใช้ `@executor(id="...")` เพื่อสร้างองค์ประกอบของเวิร์กโฟลว์
   - Executor สามารถแปลงข้อมูลหรือทำผลข้างเคียงได้
   - ใช้ `ctx.yield_output()` เพื่อผลิตผลลัพธ์ของเวิร์กโฟลว์

### 🚀 การใช้งานจริง:

- **การจองการเดินทาง**: ตรวจสอบความพร้อมใช้งาน, แนะนำทางเลือก, เปรียบเทียบตัวเลือก
- **บริการลูกค้า**: กำหนดเส้นทางตามประเภทปัญหา, ความรู้สึก, ลำดับความสำคัญ
- **อีคอมเมิร์ซ**: ตรวจสอบสต็อก, แนะนำทางเลือก, ประมวลผลคำสั่งซื้อ
- **การตรวจสอบเนื้อหา**: กำหนดเส้นทางตามคะแนนความเป็นพิษ, ธงผู้ใช้
- **เวิร์กโฟลว์อนุมัติ**: กำหนดเส้นทางตามจำนวนเงิน, บทบาทผู้ใช้, ระดับความเสี่ยง
- **การประมวลผลหลายขั้นตอน**: กำหนดเส้นทางตามคุณภาพข้อมูล, ความสมบูรณ์

### 📚 ขั้นตอนถัดไป:

- เพิ่มเงื่อนไขที่ซับซ้อนมากขึ้น (หลายเกณฑ์)
- นำลูปมาใช้พร้อมการจัดการสถานะเวิร์กโฟลว์
- เพิ่มเวิร์กโฟลว์ย่อยสำหรับส่วนประกอบที่นำกลับมาใช้ใหม่ได้
- ผสานกับ API จริง (การจองโรงแรม, ระบบสต็อก)
- เพิ่มการจัดการข้อผิดพลาดและเส้นทางสำรอง
- แสดงภาพเวิร์กโฟลว์ด้วยเครื่องมือแสดงภาพในตัว


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
